In [2]:
import pandas as pd
import os

# Folder path
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

files = {
    "users": "users.csv",
    "friends": "friends.csv",
    "user_games": "user_games.csv",
    "games": "games.csv"
}

dfs = {}

# -------------------
# LOAD FILES + ROW COUNTS
# -------------------
for name, file in files.items():
    path = os.path.join(data_path, file)
    df = pd.read_csv(path)
    dfs[name] = df

        # --- CLEANUP FOR USERS FILE ---
    if name == "users":
        # Remove useless column
        df.drop(columns=["last_logoff"], inplace=True, errors="ignore")

        # Convert account creation timestamp → datetime
        if "account_created" in df.columns:
            df["account_created"] = pd.to_datetime(
                df["account_created"],
                unit="s",
                errors="coerce"
            )

    dfs[name] = df
    print(f"\n===== {file} =====")
    print(df.head())
    print(f"{file}: {len(df):,} rows")


# -------------------
# DISCOVERED VS VISITED USERS
# -------------------
print("\n===== USER COUNTS =====")

# visited = users we actually crawled
visited_users = len(dfs["users"])
print(f"Visited users: {visited_users:,}")

# discovered ≈ anyone appearing in friend edges
friends_df = dfs["friends"]
discovered_users = pd.unique(
    pd.concat([friends_df["user_steamid"], friends_df["friend_steamid"]])
)

print(f"Discovered users (unique IDs in network): {len(discovered_users):,}")


# -------------------
# MISSING VALUES PER FILE
# -------------------
print("\n===== MISSING VALUES =====")
for name, df in dfs.items():
    print(f"\n{name}.csv")
    print(df.isna().sum())


# -------------------
# PUBLIC VS PRIVATE ACCOUNTS
# -------------------
print("\n===== ACCOUNT VISIBILITY =====")

users_df = dfs["users"]

if "public" in users_df.columns:
    public_counts = users_df["public"].value_counts(dropna=False)

    public = public_counts.get(True, 0)
    private = public_counts.get(False, 0)
    total = public + private

    print(f"Public accounts : {public:,} ({public/total:.2%})")
    print(f"Private accounts: {private:,} ({private/total:.2%})")
else:
    print("Column 'public' not found in users.csv")


===== users.csv =====
             steamid  public country     account_created
0  76561198064675174    True      IT 2012-06-02 17:21:29
1  76561198109425210    True      BE 2013-10-02 15:04:27
2  76561198281198045    True     NaN 2016-02-03 14:51:35
3  76561197994839969    True      BE 2007-12-12 16:48:56
4  76561197972380369    True      IT 2004-12-25 22:59:33
users.csv: 99,924 rows

===== friends.csv =====
        user_steamid     friend_steamid
0  76561198064675174  76561197967152992
1  76561198064675174  76561197968602157
2  76561198064675174  76561197970310193
3  76561198064675174  76561197970551918
4  76561198064675174  76561197970583217
friends.csv: 8,039,785 rows

===== user_games.csv =====
             steamid  appid  playtime_minutes
0  76561198064675174    400               367
1  76561198064675174    500               117
2  76561198064675174    550                 0
3  76561198064675174    620               710
4  76561198064675174   3900              7075
user_games.csv:

In [2]:
# Fill missing country values with "Unknown" just for clarity
users_df["country"] = users_df["country"].fillna("Unknown")

# Exclude "Unknown" for stats
known_countries = users_df[users_df["country"] != "Unknown"]

# Counts and percentages (excluding Unknown)
country_counts = known_countries["country"].value_counts()
country_percent = (known_countries["country"]
                   .value_counts(normalize=True)
                   .mul(100)
                   .round(2))

# Create dataframe
country_stats = pd.DataFrame({
    "count": country_counts,
    "percent": country_percent
})

# ------------------------
# Top 10 countries
# ------------------------
top_10_countries = country_stats.head(10)
print("=== Top 10 Countries ===")
display(top_10_countries)

# ------------------------
# Full list of countries ordered by count
# ------------------------
all_countries_sorted = country_stats.sort_values("count", ascending=False)
print("\n=== All Countries Sorted by Count ===")
display(all_countries_sorted)

=== Top 10 Countries ===


,count,percent
country,,
US,6527,11.22
CN,5060,8.70
BR,3915,6.73
RU,3511,6.04
DE,3040,5.23
PL,2230,3.83
GB,1851,3.18
TR,1780,3.06
FR,1709,2.94



=== All Countries Sorted by Count ===


,count,percent
country,,
US,6527,11.22
CN,5060,8.70
BR,3915,6.73
RU,3511,6.04
DE,3040,5.23
...,...,...
MW,2,0.00
MR,2,0.00
RE,2,0.00


In [3]:
# Exclude "Unknown"
known_countries = users_df[users_df["country"] != "Unknown"]

# List of countries ordered by count
all_countries_list = known_countries["country"].value_counts().index.tolist()

# Display in a single line
print(all_countries_list)

# Count how many users are from Italy
italy_count = (users_df["country"] == "IT").sum()  # Steam uses ISO codes
print(f"\nNumber of users from Italy: {italy_count}")

['US', 'CN', 'BR', 'RU', 'DE', 'PL', 'GB', 'TR', 'FR', 'PT', 'CA', 'JP', 'SE', 'IN', 'RO', 'DK', 'ID', 'NL', 'AU', 'CZ', 'AR', 'IT', 'NO', 'UA', 'ES', 'BE', 'FI', 'RS', 'PH', 'VN', 'SK', 'CH', 'MY', 'TW', 'HK', 'AT', 'MX', 'TH', 'KZ', 'HU', 'NE', 'KR', 'IL', 'PE', 'GR', 'SG', 'LT', 'IS', 'CL', 'BY', 'BG', 'AE', 'HR', 'ZA', 'NZ', 'PK', 'AQ', 'AF', 'EE', 'LV', 'CC', 'VA', 'CO', 'ZW', 'IE', 'CX', 'JM', 'KP', 'VE', 'BA', 'AL', 'IR', 'CG', 'UM', 'SA', 'MK', 'FJ', 'UG', 'AZ', 'SI', 'CU', 'CK', 'GE', 'UY', 'MT', 'PS', 'VI', 'IQ', 'GS', 'LB', 'HM', 'MN', 'CR', 'UZ', 'MC', 'EG', 'IM', 'DZ', 'SO', 'BD', 'TD', 'NG', 'MA', 'EC', 'QA', 'SY', 'CY', 'CD', 'JO', 'LU', 'LK', 'MD', 'TN', 'KG', 'PR', 'MG', 'ME', 'MM', 'XK', 'AM', 'FO', 'BS', 'CF', 'VG', 'AS', 'NP', 'GL', 'DO', 'DJ', 'MV', 'AO', 'KH', 'WF', 'SV', 'MZ', 'GT', 'KE', 'ET', 'KW', 'KY', 'TG', 'PG', 'MO', 'SM', 'AX', 'BN', 'BQ', 'BM', 'TJ', 'SC', 'PA', 'BF', 'KN', 'GH', 'OM', 'YE', 'BO', 'NF', 'LI', 'VC', 'ZM', 'MU', 'FM', 'PY', 'BW', 'NU', 'GU

US → United States

CN → China

BR → Brazil

RU → Russia

DE → Germany

PL → Poland

GB → United Kingdom

TR → Turkey

FR → France

PT → Portugal

CA → Canada

JP → Japan

SE → Sweden

IN → India

RO → Romania

DK → Denmark

ID → Indonesia

NL → Netherlands

AU → Australia

CZ → Czech Republic

AR → Argentina

IT → Italy

NO → Norway

UA → Ukraine

ES → Spain

BE → Belgium

FI → Finland

RS → Serbia

PH → Philippines

VN → Vietnam

SK → Slovakia

CH → Switzerland

MY → Malaysia

TW → Taiwan

HK → Hong Kong

AT → Austria

MX → Mexico

TH → Thailand

KZ → Kazakhstan

HU → Hungary

NE → Niger

KR → South Korea

IL → Israel

PE → Peru

GR → Greece

SG → Singapore

LT → Lithuania

IS → Iceland

CL → Chile

BY → Belarus

BG → Bulgaria

AE → United Arab Emirates

HR → Croatia

ZA → South Africa

NZ → New Zealand

PK → Pakistan

AQ → Antarctica

AF → Afghanistan

EE → Estonia

LV → Latvia

CC → Cocos (Keeling) Islands

VA → Vatican City

CO → Colombia

ZW → Zimbabwe

IE → Ireland

CX → Christmas Island

JM → Jamaica

KP → North Korea

VE → Venezuela

BA → Bosnia and Herzegovina

AL → Albania

IR → Iran

CG → Republic of the Congo

UM → United States Minor Outlying Islands

SA → Saudi Arabia

MK → North Macedonia

FJ → Fiji

UG → Uganda

AZ → Azerbaijan

SI → Slovenia

CU → Cuba

CK → Cook Islands

GE → Georgia

UY → Uruguay

MT → Malta

PS → Palestine

VI → U.S. Virgin Islands

IQ → Iraq

GS → South Georgia and South Sandwich Islands

LB → Lebanon

HM → Heard Island and McDonald Islands

MN → Mongolia

CR → Costa Rica

UZ → Uzbekistan

MC → Monaco

EG → Egypt

IM → Isle of Man

DZ → Algeria

SO → Somalia

BD → Bangladesh

TD → Chad

NG → Nigeria

MA → Morocco

EC → Ecuador

QA → Qatar

SY → Syria

CY → Cyprus

CD → Democratic Republic of the Congo

JO → Jordan

LU → Luxembourg

LK → Sri Lanka

MD → Moldova

TN → Tunisia

KG → Kyrgyzstan

PR → Puerto Rico

MG → Madagascar

ME → Montenegro

MM → Myanmar

XK → Kosovo

AM → Armenia

FO → Faroe Islands

BS → Bahamas

CF → Central African Republic

VG → British Virgin Islands

AS → American Samoa

NP → Nepal

GL → Greenland

DO → Dominican Republic

DJ → Djibouti

MV → Maldives

AO → Angola

KH → Cambodia

WF → Wallis and Futuna

SV → El Salvador

MZ → Mozambique

GT → Guatemala

KE → Kenya

ET → Ethiopia

KW → Kuwait

KY → Cayman Islands

TG → Togo

PG → Papua New Guinea

MO → Macau

SM → San Marino

AX → Åland Islands

BN → Brunei

BQ → Bonaire

BM → Bermuda

TJ → Tajikistan

SC → Seychelles

PA → Panama

BF → Burkina Faso

KN → Saint Kitts and Nevis

GH → Ghana

OM → Oman

YE → Yemen

BO → Bolivia

NF → Norfolk Island

LI → Liechtenstein

VC → Saint Vincent and the Grenadines

ZM → Zambia

MU → Mauritius

FM → Micronesia

PY → Paraguay

BW → Botswana

NU → Niue

GU → Guam

AG → Antigua and Barbuda

NI → Nicaragua

FK → Falkland Islands

HN → Honduras

TT → Trinidad and Tobago

BT → Bhutan

CW → Curaçao

TC → Turks and Caicos Islands

MF → Saint Martin

HT → Haiti

GI → Gibraltar

EH → Western Sahara

VU → Vanuatu

AD → Andorra

TM → Turkmenistan

LC → Saint Lucia

KI → Kiribati

PM → Saint Pierre and Miquelon

TK → Tokelau

LY → Libya

SL → Sierra Leone

CM → Cameroon

GP → Guadeloupe

TL → Timor-Leste

SZ → Eswatini

SD → Sudan

CI → Côte d’Ivoire

LR → Liberia

JE → Jersey

PN → Pitcairn Islands

GA → Gabon

LA → Laos

FX → Metropolitan France (old code, rarely used)

BZ → Belize

SJ → Svalbard and Jan Mayen

SH → Saint Helena

BB → Barbados

NR → Nauru

TZ → Tanzania

CV → Cape Verde

SS → South Sudan

AW → Aruba

BL → Saint Barthélemy

MP → Northern Mariana Islands

TO → Tonga

WS → Samoa

SX → Sint Maarten

BI → Burundi

NC → New Caledonia

SB → Solomon Islands

BH → Bahrain

PW → Palau

GG → Guernsey

DM → Dominica

IO → British Indian Ocean Territory

TF → French Southern and Antarctic Lands

KM → Comoros

YT → Mayotte

ER → Eritrea

MH → Marshall Islands

RW → Rwanda

ST → São Tomé and Príncipe

BV → Bouvet Island

AI → Anguilla

TV → Tuvalu

GQ → Equatorial Guinea

MS → Montserrat

GW → Guinea-Bissau

YU → Yugoslavia (old code, now replaced by RS/ME)

GM → Gambia

SN → Senegal

PF → French Polynesia

SR → Suriname

GN → Guinea

ML → Mali

GF → French Guiana

BJ → Benin

MW → Malawi

MR → Mauritania

RE → Réunion

MQ → Martinique

GD → Grenada

In [5]:
print("Oldest account:")
print(users_df["account_created"].min())

print("\nNewest account:")
print(users_df["account_created"].max())

Oldest account:
2003-09-11 01:29:16

Newest account:
2026-02-04 12:00:52


In [6]:
users_df["creation_year"] = users_df["account_created"].dt.year
users_df["creation_month"] = users_df["account_created"].dt.to_period("M")

year_counts = users_df["creation_year"].value_counts().sort_index()
month_counts = users_df["creation_month"].value_counts().sort_index()

year_counts

creation_year
2003.0      652
2004.0      923
2005.0      664
2006.0      760
2007.0     1197
2008.0     1403
2009.0     2160
2010.0     2585
2011.0     3731
2012.0     5488
2013.0     8210
2014.0     9921
2015.0    11209
2016.0     9484
2017.0     8295
2018.0     5293
2019.0     4568
2020.0     5085
2021.0     2944
2022.0     1959
2023.0     1515
2024.0     1022
2025.0      610
2026.0       22
Name: count, dtype: int64

In [7]:
# Ensure account_created is datetime
users_df["account_created"] = pd.to_datetime(users_df["account_created"], errors="coerce")

# Extract month number (1=Jan, 2=Feb, ..., 12=Dec)
users_df["creation_month_num"] = users_df["account_created"].dt.month

# Map month numbers to names
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
users_df["creation_month_name"] = users_df["creation_month_num"].map(month_names)

# Count users per month across all years
month_counts = users_df["creation_month_name"].value_counts().reindex(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
)

print("Account creations by month (all years combined):")
print(month_counts)

Account creations by month (all years combined):
creation_month_name
Jan    7925
Feb    6913
Mar    7259
Apr    6632
May    6516
Jun    7881
Jul    7767
Aug    7502
Sep    7037
Oct    7029
Nov    7645
Dec    9594
Name: count, dtype: int64


### Steam Friends Facts 
* base limit of friends: 250
* leveling Steam profile increases the cap by 5 friends for every level earned up to 2000
* friends are mutual

https://help.steampowered.com/en/faqs/view/1902-2D54-E445-59DD

In [13]:
# Number of unique users in user_games.csv
unique_users = user_games_df["steamid"].nunique()
print(f"Number of unique users in user_games.csv: {unique_users:,}")

Number of unique users in user_games.csv: 49,220


In [15]:
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"
user_games_df = pd.read_csv(os.path.join(data_path, "user_games.csv"))

# Convert minutes → hours
user_games_df["playtime_hours"] = user_games_df["playtime_minutes"] / 60

# -------------------
# Users with at least one unplayed game
# -------------------
users_with_unplayed = user_games_df[user_games_df["playtime_minutes"] == 0]["steamid"].nunique()
print(f"Number of users with at least one game they never played: {users_with_unplayed:,}")

# -------------------
# Games per user
# -------------------
games_per_user = user_games_df.groupby("steamid")["appid"].count()
print("\nGames per user:")
print(f"Min: {games_per_user.min()} games")
print(f"Median: {games_per_user.median():.0f} games")
print(f"Max: {games_per_user.max()} games")

# -------------------
# Total playtime per user
# -------------------
total_playtime_per_user = user_games_df.groupby("steamid")["playtime_hours"].sum()
print("\nTotal playtime per user (hours):")
print(f"Min: {total_playtime_per_user.min():,.2f} h")
print(f"Median: {total_playtime_per_user.median():,.2f} h")
print(f"Max: {total_playtime_per_user.max():,.2f} h")

# Top 10 users by total playtime
top_users = total_playtime_per_user.sort_values(ascending=False).head(10)
print("\nTop 10 users by total playtime (hours):")
for steamid, hours in top_users.items():
    print(f"{steamid}: {hours:,.2f} h")

# -------------------
# Total playtime per game
# -------------------
total_playtime_per_game = user_games_df.groupby("appid")["playtime_hours"].sum()
print("\nTotal playtime per game (hours):")
print(f"Min: {total_playtime_per_game.min():,.2f} h")
print(f"Median: {total_playtime_per_game.median():,.2f} h")
print(f"Max: {total_playtime_per_game.max():,.2f} h")

# Top 10 games by total playtime
top_games = total_playtime_per_game.sort_values(ascending=False).head(10)
print("\nTop 10 games by total playtime (hours):")
for appid, hours in top_games.items():
    print(f"{appid}: {hours:,.2f} h")

Number of users with at least one game they never played: 45,635

Games per user:
Min: 1 games
Median: 56 games
Max: 38420 games

Total playtime per user (hours):
Min: 0.00 h
Median: 2,988.62 h
Max: 2,266,794.75 h

Top 10 users by total playtime (hours):
76561198152959083: 2,266,794.75 h
76561198193401154: 2,265,379.78 h
76561198199186378: 1,408,437.80 h
76561198844100946: 1,232,288.97 h
76561198339649191: 998,464.75 h
76561198135380739: 723,297.28 h
76561198858160139: 676,861.88 h
76561198276335578: 637,090.22 h
76561197992597660: 576,246.62 h
76561198863249050: 544,268.52 h

Total playtime per game (hours):
Min: 0.00 h
Median: 26.55 h
Max: 54,895,210.95 h

Top 10 games by total playtime (hours):
730: 54,895,210.95 h
570: 14,431,474.32 h
578080: 5,876,129.02 h
271590: 5,177,553.10 h
252490: 5,049,801.07 h
440: 4,707,818.80 h
4000: 4,217,652.42 h
252950: 4,048,422.55 h
359550: 3,946,784.25 h
10: 2,994,765.62 h


In [20]:
friends_df = dfs["friends"]

# Unique users who have at least one friend recorded
unique_users = friends_df["user_steamid"].nunique()

print(f"Number of unique users in friends.csv (user_steamid): {unique_users:,}")

Number of unique users in friends.csv (user_steamid): 78,930


In [21]:
import pandas as pd
import networkx as nx

# Load friends dataframe
friends_df = dfs["friends"]

# -------------------
# 1. Basic counts
# -------------------
num_edges = len(friends_df)
unique_users = friends_df["user_steamid"].nunique()  # users we expanded
all_nodes = pd.unique(friends_df[["user_steamid", "friend_steamid"]].values.ravel())
num_nodes = len(all_nodes)

print(f"Number of nodes (unique users in edges): {num_nodes:,}")
print(f"Number of edges (friendships): {num_edges:,}")

# -------------------
# 2. Build NetworkX graph
# -------------------
G = nx.Graph()
# add all edges as undirected
edges = list(friends_df.itertuples(index=False, name=None))
G.add_edges_from(edges)

# -------------------
# 3. Degree statistics
# -------------------
degrees = [d for n, d in G.degree()]
avg_degree = sum(degrees) / len(degrees)
min_degree = min(degrees)
max_degree = max(degrees)
median_degree = int(pd.Series(degrees).median())

print("\nDegree statistics:")
print(f"Average degree: {avg_degree:.2f}")
print(f"Median degree: {median_degree}")
print(f"Min degree: {min_degree}")
print(f"Max degree: {max_degree}")

# -------------------
# 4. Density
# -------------------
density = nx.density(G)
print(f"\nNetwork density: {density:.6f}")

# -------------------
# 5. Connected components
# -------------------
components = list(nx.connected_components(G))
num_components = len(components)
largest_component_size = len(max(components, key=len))

print(f"\nNumber of connected components: {num_components}")
print(f"Largest connected component size: {largest_component_size:,}")

# -------------------
# 6. Average clustering coefficient
# -------------------
avg_clustering = nx.average_clustering(G)
print(f"\nAverage clustering coefficient: {avg_clustering:.4f}")

Number of nodes (unique users in edges): 6,084,524
Number of edges (friendships): 8,039,785

Degree statistics:
Average degree: 2.61
Median degree: 1
Min degree: 1
Max degree: 1999

Network density: 0.000000

Number of connected components: 1
Largest connected component size: 6,084,524

Average clustering coefficient: 0.0313


In [17]:
# Path to your enriched_games.csv
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"
enriched_path = os.path.join(data_path, "enriched_games.csv")

# Load the CSV properly
enriched_games_df = pd.read_csv(enriched_path, sep=";")

# Show first rows nicely
pd.set_option("display.max_columns", None)  # show all columns
pd.set_option("display.width", 120)         # adjust width for readability

print("\n===== enriched_games.csv =====")
print(enriched_games_df.head())
print(f"enriched_games.csv: {len(enriched_games_df):,} rows")


===== enriched_games.csv =====
     appid                         name                                             genres  \
0      730             Counter-Strike 2                                Action;Free To Play   
1      550                Left 4 Dead 2                                             Action   
2   218620                     PAYDAY 2                                         Action;RPG   
3   431960             Wallpaper Engine  Casual;Indie;Animation & Modeling;Design & Ill...   
4  3240220  Grand Theft Auto V Enhanced                            Action;Adventure;Racing   

                         developer                 publisher      platforms  release_date  user_count  
0                            Valve                     Valve  windows;linux  21 Aug, 2012       38752  
1                            Valve                     Valve  windows;linux  16 Nov, 2009       26253  
2  OVERKILL - a Starbreeze Studio.  Starbreeze Entertainment  windows;linux  13 Aug, 2013  

In [21]:
# Extract user_games
user_games_df = dfs["user_games"]

# -----------------------------
# Clean column names
# -----------------------------
enriched_games_df.columns = enriched_games_df.columns.str.strip()
user_games_df.columns = user_games_df.columns.str.strip()

# Ensure user_count is numeric and fill missing values
enriched_games_df["user_count"] = pd.to_numeric(enriched_games_df["user_count"], errors="coerce").fillna(0)

# -----------------------------
# Top 15 games by user_count
# -----------------------------
top15_users = enriched_games_df.sort_values("user_count", ascending=False).head(15)
print("Top 15 games by number of players:")
print(top15_users[["appid", "name", "user_count"]])

# -----------------------------
# Top 15 games by total playtime
# -----------------------------
# Compute total playtime per game from user_games
total_playtime = user_games_df.groupby("appid")["playtime_minutes"].sum().reset_index()
total_playtime.rename(columns={"playtime_minutes": "total_playtime_minutes"}, inplace=True)

# Merge with enriched games
games_playtime_df = enriched_games_df.merge(total_playtime, on="appid", how="left")
games_playtime_df["total_playtime_minutes"] = games_playtime_df["total_playtime_minutes"].fillna(0)

top15_playtime = games_playtime_df.sort_values("total_playtime_minutes", ascending=False).head(15)
print("\nTop 15 games by total playtime:")
print(top15_playtime[["appid", "name", "total_playtime_minutes"]])

# -----------------------------
# Min, Max, Median of users counted
# -----------------------------
min_users = enriched_games_df["user_count"].min()
max_users = enriched_games_df["user_count"].max()
median_users = enriched_games_df["user_count"].median()
print(f"\nUsers counted per game: min={min_users}, max={max_users}, median={median_users}")

# -----------------------------
# Number of games with zero playtime
# -----------------------------
zero_playtime_games = (games_playtime_df["total_playtime_minutes"] == 0).sum()
print(f"\nNumber of games with zero playtime: {zero_playtime_games}")

# -----------------------------
# List of all genres and top genres
# -----------------------------
# Split genres into lists
enriched_games_df["genres_list"] = enriched_games_df["genres"].str.split(";")

# Flatten all genres
all_genres = [genre for sublist in enriched_games_df["genres_list"].dropna() for genre in sublist]
genres_series = pd.Series(all_genres)

# Top genres
top_genres = genres_series.value_counts()
print("\nTop genres:")
print(top_genres.head(15))

# -----------------------------
# Number of games published each year
# -----------------------------
enriched_games_df["release_date"] = pd.to_datetime(enriched_games_df["release_date"], errors="coerce")
games_per_year = enriched_games_df.groupby(enriched_games_df["release_date"].dt.year)["appid"].count().sort_index()
print("\nNumber of games published each year:")
print(games_per_year.tail(15))  # last 15 years for brevity

# -----------------------------
# Number of games per platform
# -----------------------------
# Split platforms into lists
enriched_games_df["platforms_list"] = enriched_games_df["platforms"].str.split(";")
all_platforms = [p for sublist in enriched_games_df["platforms_list"].dropna() for p in sublist]
platform_counts = pd.Series(all_platforms).value_counts()
print("\nNumber of games per platform:")
print(platform_counts)

# -----------------------------
# Top developer
# -----------------------------
print("")
top_dev_df = enriched_games_df["developer"].value_counts().head(10).reset_index()
top_dev_df.columns = ["Developer", "Number_of_Games"]
print(top_dev_df)

Top 15 games by number of players:
      appid                               name  user_count
0       730                   Counter-Strike 2       38752
1       550                      Left 4 Dead 2       26253
2    218620                           PAYDAY 2       24022
3    431960                   Wallpaper Engine       21856
4   3240220        Grand Theft Auto V Enhanced       19712
5    271590          Grand Theft Auto V Legacy       19681
6    407530   ARK: The Survival Of The Fittest       18439
7    346110              ARK: Survival Evolved       17857
8      4000                        Garry's Mod       17711
9    578080                PUBG: BATTLEGROUNDS       17580
10      620                           Portal 2       17020
11   945360                           Among Us       16613
12   242760                         The Forest       16435
13   359550  Tom Clancy's Rainbow Six® Siege X       16270
14   105600                           Terraria       16228

Top 15 games by tota

## Filtering

In [1]:
import pandas as pd
import os

# ------------------------------------
# PATH
# ------------------------------------
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

# ------------------------------------
# LOAD DATA
# ------------------------------------
users_df = pd.read_csv(os.path.join(data_path, "users.csv"))
friends_df = pd.read_csv(os.path.join(data_path, "friends.csv"))
user_games_df = pd.read_csv(os.path.join(data_path, "user_games.csv"))

# ------------------------------------
# Public users only
# ------------------------------------
public_users = users_df[users_df["public"] == True]["steamid"]

# ------------------------------------
# Users appearing as crawled users in friends.csv
# (only user_steamid — not friend_steamid)
# ------------------------------------
friends_users = friends_df["user_steamid"].unique()

# ------------------------------------
# Users present in user_games.csv
# ------------------------------------
game_users = user_games_df["steamid"].unique()

# ------------------------------------
# INTERSECTION = valid users
# ------------------------------------
valid_users = set(public_users) & set(friends_users) & set(game_users)

print(f"Valid users after filtering: {len(valid_users):,}")

# ------------------------------------
# FILTER DATASETS
# ------------------------------------
filtered_users = users_df[users_df["steamid"].isin(valid_users)]
filtered_friends = friends_df[friends_df["user_steamid"].isin(valid_users)]
filtered_games = user_games_df[user_games_df["steamid"].isin(valid_users)]

# ------------------------------------
# SAVE NEW CSV FILES
# ------------------------------------
filtered_users.to_csv(os.path.join(data_path, "users_filtered.csv"), index=False)
filtered_friends.to_csv(os.path.join(data_path, "friends_filtered.csv"), index=False)
filtered_games.to_csv(os.path.join(data_path, "user_games_filtered.csv"), index=False)

print("Filtered files saved successfully.")

Valid users after filtering: 43,675
Filtered files saved successfully.


In [2]:
import pandas as pd
import os

data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

files = {
    "users_filtered": "users_filtered.csv",
    "friends_filtered": "friends_filtered.csv",
    "user_games_filtered": "user_games_filtered.csv"
}

for name, file in files.items():
    path = os.path.join(data_path, file)
    df = pd.read_csv(path)

    print(f"\n===== {file} =====")
    print(f"Rows: {len(df):,}")

    if name == "users_filtered":
        print(f"Unique users: {df['steamid'].nunique():,}")

    elif name == "friends_filtered":
        print(f"Unique user_steamid: {df['user_steamid'].nunique():,}")
        print(f"Unique friend_steamid: {df['friend_steamid'].nunique():,}")

    elif name == "user_games_filtered":
        print(f"Unique users: {df['steamid'].nunique():,}")
        print(f"Unique games: {df['appid'].nunique():,}")


===== users_filtered.csv =====
Rows: 43,675
Unique users: 43,675

===== friends_filtered.csv =====
Rows: 5,396,360
Unique user_steamid: 43,675
Unique friend_steamid: 4,253,328

===== user_games_filtered.csv =====
Rows: 7,740,459
Unique users: 43,675
Unique games: 44,846


In [4]:
import pandas as pd
import os

data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

# Load datasets
user_games_df = pd.read_csv(os.path.join(data_path, "user_games.csv"))
enriched_games_df = pd.read_csv(
    os.path.join(data_path, "enriched_games.csv"),
    sep=";",
    engine="python"
)

# Unique appids
user_games_unique = set(user_games_df["appid"].unique())
enriched_unique = set(enriched_games_df["appid"].unique())

missing = user_games_unique - enriched_unique

print("Unique games in user_games:", len(user_games_unique))
print("Unique games in enriched_games:", len(enriched_unique))
print("Games missing metadata:", len(missing))

Unique games in user_games: 45049
Unique games in enriched_games: 21052
Games missing metadata: 23997


In [10]:
import pandas as pd
import os

# Path to your data folder
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

# Load filtered user_games
user_games_filtered_path = os.path.join(data_path, "user_games_filtered.csv")
user_games_filtered = pd.read_csv(user_games_filtered_path)

# Load enriched games
enriched_games_path = os.path.join(data_path, "enriched_games.csv")
enriched_games_df = pd.read_csv(enriched_games_path, sep=';')

# ---------------------------
# Filter user_games to only keep games present in enriched_games
# ---------------------------
user_games_filtered_enriched = user_games_filtered[
    user_games_filtered["appid"].isin(enriched_games_df["appid"])
]

# ---------------------------
# Compare counts
# ---------------------------
print("===== Comparison with enriched_games.csv =====")
print(f"Rows in filtered user_games (after keeping only enriched games): {len(user_games_filtered_enriched):,}")
print(f"Unique games in filtered user_games: {user_games_filtered_enriched['appid'].nunique():,}")
print(f"Games in enriched_games.csv: {enriched_games_df['appid'].nunique():,}")

# ---------------------------
# Save filtered CSV
# ---------------------------
filtered_path = os.path.join(data_path, "user_games_filtered_all_games_with_metadata.csv")
user_games_filtered_enriched.to_csv(filtered_path, index=False)
print(f"\nFiltered CSV saved to: {filtered_path}")

===== Comparison with enriched_games.csv =====
Rows in filtered user_games (after keeping only enriched games): 4,903,436
Unique games in filtered user_games: 20,988
Games in enriched_games.csv: 21,052

Filtered CSV saved to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final\user_games_filtered_all_games_with_metadata.csv


In [11]:
import pandas as pd
import os

# ------------------------
# Paths
# ------------------------
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"

user_games_filtered_path = os.path.join(data_path, "user_games_filtered.csv")
enriched_games_path = os.path.join(data_path, "enriched_games.csv")

# Load the filtered user_games
user_games_filtered = pd.read_csv(user_games_filtered_path)

# Load enriched_games (with correct separator)
enriched_games_df = pd.read_csv(enriched_games_path, sep=';')

# ------------------------
# Keep only enriched games present in filtered user_games
# ------------------------
enriched_games_filtered = enriched_games_df[
    enriched_games_df["appid"].isin(user_games_filtered["appid"])
]

# Save to CSV
enriched_games_filtered_path = os.path.join(data_path, "enriched_games_filtered.csv")
enriched_games_filtered.to_csv(enriched_games_filtered_path, index=False, sep=';')

# ------------------------
# Quick check
# ------------------------
print("===== Filtered enriched_games.csv =====")
print(f"Rows: {len(enriched_games_filtered):,}")
print(f"Unique games: {enriched_games_filtered['appid'].nunique():,}")

===== Filtered enriched_games.csv =====
Rows: 20,988
Unique games: 20,988


In [14]:
import pandas as pd
import os

# ------------------------
# Paths
# ------------------------
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\checkpoint_export_final"
user_games_filtered_path = os.path.join(data_path, "user_games_filtered_all_games_with_metadata.csv")
enriched_games_filtered_path = os.path.join(data_path, "enriched_games_filtered.csv")

# Load filtered files
user_games_filtered = pd.read_csv(user_games_filtered_path)
enriched_games_filtered = pd.read_csv(enriched_games_filtered_path, sep=';')

# ------------------------
# Check 1: All games in user_games_filtered are in enriched_games_filtered
# ------------------------
user_games_set = set(user_games_filtered["appid"].unique())
enriched_games_set = set(enriched_games_filtered["appid"].unique())

missing_in_enriched = user_games_set - enriched_games_set
missing_in_user_games = enriched_games_set - user_games_set

print("===== Sanity Check =====")
print(f"Unique games in user_games_filtered: {len(user_games_set):,}")
print(f"Unique games in enriched_games_filtered: {len(enriched_games_set):,}")

if missing_in_enriched:
    print(f"WARNING: {len(missing_in_enriched)} games in user_games_filtered are missing in enriched_games_filtered")
else:
    print("✅ All games in user_games_filtered_all_games_with_metadata are present in enriched_games_filtered")

if missing_in_user_games:
    print(f"NOTE: {len(missing_in_user_games)} games in enriched_games_filtered are not owned by any user")
else:
    print("✅ All games in enriched_games_filtered are owned by at least one user")

# Optional: print a few missing IDs for inspection
if missing_in_enriched:
    print("Example missing game IDs:", list(missing_in_enriched)[:10])
if missing_in_user_games:
    print("Example unowned game IDs:", list(missing_in_user_games)[:10])

===== Sanity Check =====
Unique games in user_games_filtered: 20,988
Unique games in enriched_games_filtered: 20,988
✅ All games in user_games_filtered_all_games_with_metadata are present in enriched_games_filtered
✅ All games in enriched_games_filtered are owned by at least one user


## CODE TO PREPARE DATA FOR MICHELE'S ALGORITHM

In [22]:
import pandas as pd
import networkx as nx
import pickle
import os

# -----------------------------
# Paths
# -----------------------------
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\filtered_data"

users_file = os.path.join(data_path, "users_filtered.csv")
friends_file = os.path.join(data_path, "friends_filtered.csv")
user_games_file = os.path.join(data_path, "user_games_filtered_all_games_with_metadata.csv")
enriched_games_file = os.path.join(data_path, "enriched_games_filtered.csv")

output_graph_file = os.path.join(data_path, "G_users_graph.pkl")
output_matrix_file = os.path.join(data_path, "user_game_matrix.pkl")
output_mapping_file = os.path.join(data_path, "user_node_mapping.csv")

# -----------------------------
# Load data
# -----------------------------
users_df = pd.read_csv(users_file)
friends_df = pd.read_csv(friends_file)
user_games_df = pd.read_csv(user_games_file)
enriched_games_df = pd.read_csv(enriched_games_file, sep=";")

# -----------------------------
# Keep only public users
# -----------------------------
users_df = users_df[users_df["public"] == True]

# Keep only users present in friends and user_games
valid_user_ids = set(friends_df["user_steamid"]).intersection(user_games_df["steamid"])
users_df = users_df[users_df["steamid"].isin(valid_user_ids)].copy()

# Reindex users to numeric IDs from 0 to n-1
users_df = users_df.reset_index(drop=True)
users_df["node_id"] = users_df.index

# Build a mapping: steamid → node_id
steamid_to_node = dict(zip(users_df["steamid"], users_df["node_id"]))

# -----------------------------
# Build NetworkX graph
# -----------------------------
G = nx.Graph()
# Add numeric nodes
G.add_nodes_from(users_df["node_id"])

# Convert friend edges to numeric node IDs
edges = []
for _, row in friends_df.iterrows():
    u = row["user_steamid"]
    v = row["friend_steamid"]
    if u in steamid_to_node and v in steamid_to_node:
        edges.append((steamid_to_node[u], steamid_to_node[v]))

G.add_edges_from(edges)

# -----------------------------
# Build user × game matrix (0/1)
# -----------------------------
# Keep only games present in enriched_games
valid_game_ids = set(enriched_games_df["appid"])
user_games_df = user_games_df[user_games_df["appid"].isin(valid_game_ids)]

# Map steamid → node_id
user_games_df = user_games_df[user_games_df["steamid"].isin(steamid_to_node)]
user_games_df["node_id"] = user_games_df["steamid"].map(steamid_to_node)

# Pivot to user × game matrix (0/1)
user_game_matrix = user_games_df.pivot_table(
    index="node_id",
    columns="appid",
    aggfunc="size",
    fill_value=0
)

# Make sure all users are present (even if they have no games)
all_nodes = pd.Index(users_df["node_id"])
user_game_matrix = user_game_matrix.reindex(all_nodes, fill_value=0)

# -----------------------------
# Save outputs
# -----------------------------
with open(output_graph_file, "wb") as f:
    pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)

with open(output_matrix_file, "wb") as f:
    pickle.dump(user_game_matrix, f, pickle.HIGHEST_PROTOCOL)

# Save mapping CSV
users_df[["node_id", "steamid"]].to_csv(output_mapping_file, index=False)

# -----------------------------
# Sanity checks
# -----------------------------
print("✅ Prepared files for nvd:")
print(f"Users in graph: {G.number_of_nodes()}")
print(f"Friendship edges: {G.number_of_edges()}")
print(f"User × Game matrix shape: {user_game_matrix.shape}")
print(f"Saved graph to: {output_graph_file}")
print(f"Saved matrix to: {output_matrix_file}")
print(f"Saved node mapping to: {output_mapping_file}")

# Optional: check first few rows of matrix
print("\nFirst 5 rows of user × game matrix:")
print(user_game_matrix.head())

# Optional: check first 5 nodes in graph
print("\nFirst 5 graph nodes:", list(G.nodes)[:5])

✅ Prepared files for nvd:
Users in graph: 43614
Friendship edges: 44508
User × Game matrix shape: (43614, 20988)
Saved graph to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\filtered_data\G_users_graph.pkl
Saved matrix to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\filtered_data\user_game_matrix.pkl
Saved node mapping to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\filtered_data\user_node_mapping.csv

First 5 rows of user × game matrix:
appid    10       20       30       40       50       60       70       \
node_id                                                                  
0              0        0        0        0        0        0        0   
1              0        0        0        0        0        0        0   
2              0        0        0        0        0        0        0   
3              1        0        1        1        0        1        1   
4         

In [2]:
import pickle
import networkx as nx

graph_file = r"D:\bachelor_project\data_for_micheles_algorithm\G_users_graph.pkl"

# Load graph
with open(graph_file, "rb") as f:
    G = pickle.load(f)

# Number of connected components
num_components = nx.number_connected_components(G)
print("Connected components:", num_components)

# Size of each component (largest first)
component_sizes = sorted(
    [len(c) for c in nx.connected_components(G)],
    reverse=True
)

print("Largest component size:", component_sizes[0])
print("Top 10 component sizes:", component_sizes[:10])

Connected components: 8340
Largest component size: 31021
Top 10 component sizes: [31021, 40, 24, 21, 18, 18, 17, 17, 16, 15]


In [15]:
import pandas as pd
import networkx as nx
import os

# -------------------------
# File paths
# -------------------------
data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\filtered_data"

users_file = os.path.join(data_path, "users_filtered.csv")
friends_file = os.path.join(data_path, "friends_filtered.csv")
user_games_file = os.path.join(data_path, "user_games_filtered_all_games_with_metadata.csv")
games_file = os.path.join(data_path, "enriched_games_filtered.csv")

# Output files
users_out = os.path.join(data_path, "users_final.csv")
friends_out = os.path.join(data_path, "friends_final.csv")
user_games_out = os.path.join(data_path, "user_games_final.csv")
games_out = os.path.join(data_path, "games_final.csv")

# -------------------------
# 1. Load data
# -------------------------
print("Loading data...")
users_df = pd.read_csv(users_file)
friends_df = pd.read_csv(friends_file)
user_games_df = pd.read_csv(user_games_file)
games_df = pd.read_csv(games_file, sep=";")

print(f"Users: {len(users_df)}, Friendships: {len(friends_df)}, User-games: {len(user_games_df)}, Games: {len(games_df)}")

# -------------------------
# 2. Filter users like in matrix (public + present in friends & user-games)
# -------------------------
public_users = users_df[users_df["public"] == True]
valid_user_ids = set(friends_df["user_steamid"]).intersection(set(user_games_df["steamid"]))
users_df = public_users[public_users["steamid"].isin(valid_user_ids)].copy()

print(f"Users after matrix-like filtering: {len(users_df)}")

# -------------------------
# 3. Filter friends to these users
# -------------------------
friends_df = friends_df[
    friends_df["user_steamid"].isin(users_df["steamid"]) &
    friends_df["friend_steamid"].isin(users_df["steamid"])
].copy()

print(f"Friendships after filtering to valid users: {len(friends_df)}")

# -------------------------
# 4. Build friendship graph
# -------------------------
print("\nBuilding friendship graph...")
G = nx.Graph()
G.add_edges_from(zip(friends_df["user_steamid"], friends_df["friend_steamid"]))

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# -------------------------
# 5. Compute Largest Connected Component (LCC)
# -------------------------
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)
lcc_users = set(largest_cc)
print("Largest connected component size:", len(lcc_users))

# -------------------------
# 6. Filter USERS to LCC
# -------------------------
users_final = users_df[users_df["steamid"].isin(lcc_users)].copy()

# -------------------------
# 7. Filter FRIENDS to LCC
# -------------------------
friends_final = friends_df[
    friends_df["user_steamid"].isin(lcc_users) &
    friends_df["friend_steamid"].isin(lcc_users)
].copy()

# -------------------------
# 8. Filter USER-GAMES to LCC
# -------------------------
user_games_final = user_games_df[user_games_df["steamid"].isin(lcc_users)].copy()

# -------------------------
# 9. Filter GAMES to only those played by remaining users
# -------------------------
games_used = set(user_games_final["appid"].unique())
games_final = games_df[games_df["appid"].isin(games_used)].copy()

# -------------------------
# 10. Save final files
# -------------------------
users_final.to_csv(users_out, index=False)
friends_final.to_csv(friends_out, index=False)
user_games_final.to_csv(user_games_out, index=False)
games_final.to_csv(games_out, index=False)

# -------------------------
# 11. Sanity checks
# -------------------------
print("\n===== FINAL DATA SUMMARY =====")
print("Users:", len(users_final))
print("Friend edges:", len(friends_final))
print("User-game rows:", len(user_games_final))
print("Games:", len(games_final))

print("\nUnique users in user_games:", user_games_final["steamid"].nunique())
print("Unique games in user_games:", user_games_final["appid"].nunique())

# Check alignment
missing_in_users = set(user_games_final["steamid"].unique()) - set(users_final["steamid"].unique())
print("Users in user_games but not in users_final:", len(missing_in_users))

missing_in_friends = set(friends_final["user_steamid"].unique()).union(set(friends_final["friend_steamid"].unique())) - set(users_final["steamid"].unique())
print("Users in friends but not in users_final:", len(missing_in_friends))

missing_games = set(user_games_final["appid"].unique()) - set(games_final["appid"].unique())
print("Games in user_games but not in games_final:", len(missing_games))

print("\nDone.")

Loading data...
Users: 43675, Friendships: 5396360, User-games: 4903436, Games: 20988
Users after matrix-like filtering: 43614
Friendships after filtering to valid users: 88791

Building friendship graph...
Graph nodes: 37303
Graph edges: 44508
Largest connected component size: 31021

===== FINAL DATA SUMMARY =====
Users: 31021
Friend edges: 80211
User-game rows: 4010476
Games: 20918

Unique users in user_games: 31021
Unique games in user_games: 20918
Users in user_games but not in users_final: 0
Users in friends but not in users_final: 0
Games in user_games but not in games_final: 0

Done.


In [21]:
import pandas as pd
import os

# Folder containing final CSVs
final_data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data"

# Filenames
files = {
    "Users": "users_final.csv",
    "Friends": "friends_final.csv",
    "User-Games": "user_games_final.csv",
    "Games": "games_final.csv"
}

print("Row counts for final datasets:\n")

for name, filename in files.items():
    file_path = os.path.join(final_data_path, filename)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep="," if filename.endswith(".csv") else ";")
        print(f"{name}: {len(df)} rows")
    else:
        print(f"{name}: File not found ({filename})")

Row counts for final datasets:

Users: 31021 rows
Friends: 80211 rows
User-Games: 4010476 rows
Games: 20918 rows


In [24]:
import pandas as pd
import os

# -------------------------
# Paths
# -------------------------
raw_data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\raw_data"
final_data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data"

users_final_file = os.path.join(final_data_path, "users_final.csv")
raw_friends_file = os.path.join(raw_data_path, "friends.csv")
output_file = os.path.join(final_data_path, "users_friend_counts.csv")

# -------------------------
# Load users_final
# -------------------------
users_final = pd.read_csv(users_final_file)
print("Columns in users_final:", users_final.columns)

# Ensure column for merge is named 'steamid'
if 'steamid' not in users_final.columns:
    raise ValueError("users_final.csv does not contain a 'steamid' column!")

lcc_user_ids = set(users_final["steamid"])
print(f"Users in LCC: {len(lcc_user_ids)}")

# -------------------------
# Load raw friends.csv
# -------------------------
friends_raw = pd.read_csv(raw_friends_file)
print(f"Total rows in raw friends.csv: {len(friends_raw)}")
print("Columns in friends_raw:", friends_raw.columns)

# Ensure user column exists
if 'user_steamid' not in friends_raw.columns:
    raise ValueError("friends.csv does not contain a 'user_steamid' column!")

# -------------------------
# Keep only rows where user is in LCC (friend can be anyone)
# -------------------------
friends_raw_lcc = friends_raw[friends_raw["user_steamid"].isin(lcc_user_ids)]

# -------------------------
# Count friends for each user
# -------------------------
friend_counts = friends_raw_lcc.groupby("user_steamid").size().reset_index(name="friend_count")

# Rename to match users_final
friend_counts = friend_counts.rename(columns={"user_steamid": "steamid"})

# -------------------------
# Merge counts into users_final
# -------------------------
users_with_counts = users_final.merge(friend_counts, on="steamid", how="left")

# Users with no friends get 0
users_with_counts["friend_count"] = users_with_counts["friend_count"].fillna(0).astype(int)

# -------------------------
# Save to CSV
# -------------------------
users_with_counts.to_csv(output_file, index=False)
print(f"Saved friend counts for LCC users to: {output_file}")

# Optional: check first few rows
print(users_with_counts.head())

Columns in users_final: Index(['steamid', 'public', 'country', 'account_created', 'last_logoff'], dtype='object')
Users in LCC: 31021
Total rows in raw friends.csv: 8039785
Columns in friends_raw: Index(['user_steamid', 'friend_steamid'], dtype='object')
Saved friend counts for LCC users to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data\users_friend_counts.csv
             steamid  public country  account_created  last_logoff  \
0  76561198064675174    True      IT     1.338658e+09          NaN   
1  76561198109425210    True      BE     1.380726e+09          NaN   
2  76561198281198045    True     NaN     1.454511e+09          NaN   
3  76561197994839969    True      BE     1.197478e+09          NaN   
4  76561197972380369    True      IT     1.104016e+09          NaN   

   friend_count  
0            28  
1            16  
2            22  
3           121  
4           139  


In [30]:
import pandas as pd
import os

# Paths
final_data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data"
input_file = os.path.join(final_data_path, "users_friend_counts.csv")
output_file = os.path.join(final_data_path, "users_friend_counts_simple.csv")

# Load existing CSV
df = pd.read_csv(input_file)

# Keep only steamid and friend_count
df_simple = df[["steamid", "friend_count"]]

# Save simplified CSV
df_simple.to_csv(output_file, index=False)
print(f"Simplified friend-count CSV saved to: {output_file}")

# Optional: check first few rows
print(df_simple.head())

Simplified friend-count CSV saved to: C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data\users_friend_counts_simple.csv
             steamid  friend_count
0  76561198064675174            28
1  76561198109425210            16
2  76561198281198045            22
3  76561197994839969           121
4  76561197972380369           139


In [31]:
import pandas as pd
import os

# Path to final user-games CSV
final_data_path = r"C:\Users\jkgas\OneDrive\Desktop\data_science\Spring_2026\bachelor_project\data\final_data"
user_games_file = os.path.join(final_data_path, "user_games.csv")

# Load CSV
user_games_df = pd.read_csv(user_games_file)

# Total user-game pairs
total_pairs = len(user_games_df)
print("Total user-game pairs:", total_pairs)

# Optional: number of unique users and unique games
unique_users = user_games_df["steamid"].nunique()
unique_games = user_games_df["appid"].nunique()
print("Unique users:", unique_users)
print("Unique games:", unique_games)

Total user-game pairs: 4010476
Unique users: 31021
Unique games: 20918
